<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/01-Basic_Tutor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Packages and Setup Variables


In [ ]:
!pip install -q openai==1.107.0

In [ ]:
import os

# Set the "OPENAI_API_KEY" in the Python environment. Will be used by OpenAI client later.

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except ImportError:
    pass  # Running locally — OPENAI_API_KEY is expected in the environment

# Load the API client


In [ ]:
from openai import OpenAI

# Defining the "client" object that enables
# us to connect to OpenAI API endpoints.
client = OpenAI()

# Query the API


In [ ]:
# Define two questions: 1) Related to AI, 2) Unrelated topic.
# These questions will be used to evaluate model's performance.
QUESTION_AI = "List a number of famous artificial intelligence frameworks?"
QUESTION_NOT_AI = (
    "What is the name of the highest mountain in the world and its height?"
)

In [ ]:
def ask_ai_tutor(question):
    try:
        # System instructions for the AI tutor specialized in AI-related questions
        instructions = (
            "You are an AI tutor specialized in answering artificial intelligence-related questions. "
            "Only answer AI-related questions, else say that you cannot answer this question."
        )

        # Call the OpenAI Responses API
        response = client.responses.create(
            model="gpt-5-mini",
            reasoning={'effort':'minimal'},
            instructions=instructions,  # System-level guidance
            input=f"Please provide an informative and accurate answer to the following question.\nQuestion: {question}\nAnswer:",
        )

        # Return the AI's response using the output_text helper
        return response.output_text.strip()

    except Exception as e:
        return f"An error occurred: {e}"



In [ ]:
# Ask the AI-related question.
RES_AI = ask_ai_tutor(QUESTION_AI)
print(RES_AI)

In [ ]:
# Ask the unrelated question.
RES_NOT_AI = ask_ai_tutor(QUESTION_NOT_AI)
print(RES_NOT_AI)

# History


In [ ]:
response = client.responses.create(
    model="gpt-5-mini",
    reasoning={'effort':'minimal'},
    instructions="You are an AI tutor specialized in answering artificial intelligence-related questions. Only answer AI-related questions, else say that you cannot answer this question.",
    input=[
        {
            "role": "user",
            "content": "Please provide an informative and accurate answer to the following question.\nQuestion: List a number of famous artificial intelligence frameworks?\nAnswer:",
        },
        {
            "role": "assistant",
            "content": RES_AI
        },
        {
            "role": "user",
            "content": "Please provide an informative and accurate answer to the following question.\nQuestion: What is the name of the highest mountain in the world and its height?\nAnswer:",
        },
        {
            "role": "assistant",
            "content": RES_NOT_AI
        },
        {
            "role": "user",
            "content": "Please provide an informative and accurate answer to the following question.\nQuestion: Can you write a summary of the first suggested AI framework in the first question?\nAnswer:",
        },
    ]
)

print(response.output_text.strip())

## Optional: Streaming Responses

Instead of waiting for the full response, you can stream tokens as they are generated.
This is useful for building interactive applications where you want to show output in real time.

In [ ]:
# Stream a response token by token
def ask_ai_tutor_streaming(question):
    instructions = (
        "You are an AI tutor specialized in answering artificial intelligence-related questions. "
        "Only answer AI-related questions, else say that you cannot answer this question."
    )
    print("Streaming response:\n")
    stream = client.responses.create(
        model="gpt-5-mini",
        reasoning={"effort": "minimal"},
        instructions=instructions,
        input=f"Please provide an informative and accurate answer to the following question.\nQuestion: {question}\nAnswer:",
        stream=True,
    )
    for event in stream:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
    print()  # newline after stream ends


ask_ai_tutor_streaming("What is a neural network?")

## Optional: Persona Comparison

Compare how the AI tutor responds with different system instructions (personas).
This demonstrates how the `instructions` parameter shapes the model's tone and focus.

In [ ]:
personas = {
    "Strict Tutor": (
        "You are a strict AI tutor. Only answer AI-related questions with precise, technical detail. "
        "If the question is off-topic, firmly decline."
    ),
    "Friendly Tutor": (
        "You are a friendly AI tutor who explains concepts simply and encourages curiosity. "
        "Keep answers approachable and use everyday language."
    ),
}

test_question = "What is a neural network?"

for persona_name, instructions in personas.items():
    response = client.responses.create(
        model="gpt-5-mini",
        reasoning={"effort": "minimal"},
        instructions=instructions,
        input=f"Please provide an informative and accurate answer to the following question.\nQuestion: {test_question}\nAnswer:",
    )
    print(f"--- {persona_name} ---")
    print(response.output_text.strip())
    print()